# SiamViTRM — Fingerprint Verification with Tiny Recursive Model

This notebook trains and evaluates a **Siamese Vision Tiny Recursive Model (SiamViTRM)** for
**fingerprint verification** on FVC2004.

## What “TRM / ViTRM” means here (practically)

- **Siamese verification**: given a pair of fingerprint images 
(
\(x_1, x_2\)), we encode both with the **same** network and output a **match score**.
- **Recursive Transformer encoder (ViTRM-style)**: instead of stacking many Transformer blocks,
  we reuse a small block **recursively** with a set of **latent memory tokens** \(z\) and a
  **prediction token** \(y\). Each recursion refines \(y, z\) while sharing the same parameters.
- **TRM training recipe** (what we actually use):
  - **EMA weights** for evaluation stability (`EMA` shadow model)
  - optional **deep supervision** via repeated forward/backward passes per batch (kept at `N_SUPERVISION=1`)
  - a simple **halting head** (confidence) included in the model, but we primarily train the verification score

The verification score is computed from the final prediction-token embeddings via cosine similarity:
\[\text{score}(x_1,x_2)=\frac{\cos(v_1,v_2)+1}{2}\in[0,1].\]

## Baselines (FVC2004 test)
| Model | Params | EER |
|---|---:|---:|
| NCC (no learning) | 0 | 30.84% |
| Siamese ResNet-18 | 11.24M | 8.72% |
| **SiamViTRM (`artifacts/trm/trm_results.json`)** | **1.837M** | **15.07%** |

Notes:
- Baseline numbers come from `artifacts/baseline_results.json`.
- SiamViTRM is reported using **EMA evaluation** on **3,696** test pairs.

In [ ]:
# Imports and path setup
from __future__ import annotations

import copy
import csv
import json
import math
import random
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset

# Project paths
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRM_DIR      = ARTIFACT_DIR / "trm"
FIG_DIR      = TRM_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Import model from scripts/
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from trm_model import SiamViTRM, EMA, count_parameters  # noqa: E402

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# Device — allow CUDA/MPS but default to CPU for stability
FORCE_CPU = True
if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif not FORCE_CPU and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Image size for TRM: 128×128 gives 64 patches (16×16 each) — same patch count as ViTRM on CIFAR-10
IMG_SIZE_TRM = (128, 128)

plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRM_DIR:     ", TRM_DIR)
print("Device:      ", DEVICE)

: 

## Data Loading

Reuses the same pair CSVs generated in `01_data_engineering.ipynb`.
Images are resized to 128×128 (grayscale, single channel) before caching in memory.
This gives 64 non-overlapping 16×16 patches per image — identical patch count to
ViTRM on 32×32 CIFAR-10 with 4×4 patches.

In [ ]:
# Pair CSV loader and grayscale fingerprint dataset
def load_pairs_csv(path: Path) -> list[dict]:
    with path.open() as f:
        return list(csv.DictReader(f))


def load_gray(path: Path, size: tuple[int, int]) -> np.ndarray:
    """Load image as float32 grayscale array in [0, 1]."""
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize(size), dtype=np.float32) / 255.0


def augment_fingerprint(img: torch.Tensor) -> torch.Tensor:
    """Fingerprint augmentation (matches Colab notebook logic).

    Uses rotation + translation + brightness to simulate realistic placement/pressure.
    """
    import torchvision.transforms.functional as TF

    # Rotation — simulates finger placement angle variation
    angle = random.uniform(-15, 15)
    img = TF.rotate(img, angle, fill=0.0)

    # Translation — simulates off-center placement on sensor
    tx = random.randint(-8, 8)
    ty = random.randint(-8, 8)
    img = TF.affine(img, angle=0, translate=[tx, ty], scale=1.0, shear=0, fill=0.0)

    # Brightness — simulates finger pressure variation
    factor = 1.0 + random.uniform(-0.25, 0.25)
    img = img * factor

    return img.clamp(0.0, 1.0)


class FingerprintPairDataset(Dataset):
    """Verification pair dataset with in-memory image cache (single channel)."""

    def __init__(
        self,
        pairs_csv: Path,
        project_root: Path,
        size: tuple[int, int],
        max_pairs: int | None = None,
        augment: bool = False,
    ) -> None:
        self.project_root = project_root
        self.size = size
        self.augment = augment
        self.pairs = load_pairs_csv(pairs_csv)
        if max_pairs and len(self.pairs) > max_pairs:
            random.shuffle(self.pairs)
            self.pairs = self.pairs[:max_pairs]
        self._cache: dict[str, torch.Tensor] = {}
        self._preload()

    def _preload(self) -> None:
        uniq: set[str] = set()
        for p in self.pairs:
            uniq.add(p["image_path_1"])
            uniq.add(p["image_path_2"])
        h, w = self.size
        print(f"  Preloading {len(uniq)} images at {w}×{h} (grayscale)...")
        for i, rel in enumerate(sorted(uniq)):
            arr = load_gray(self.project_root / rel, self.size)
            self._cache[rel] = torch.from_numpy(arr).unsqueeze(0)  # (1, H, W)
            if (i + 1) % 500 == 0:
                print(f"    {i + 1}/{len(uniq)}")
        mb = len(self._cache) * h * w * 4 / 1e6
        print(f"  Cache: {len(self._cache)} images, ~{mb:.0f} MB")

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        p = self.pairs[idx]
        t1 = self._cache[p["image_path_1"]]
        t2 = self._cache[p["image_path_2"]]
        if self.augment:
            import torchvision.transforms.functional as TF
            t1 = augment_fingerprint(t1)
            t2 = augment_fingerprint(t2)
            # Apply the SAME flip decision to both images in the pair
            if random.random() > 0.5:
                t1 = TF.hflip(t1)
                t2 = TF.hflip(t2)
        label = int(p["label"])
        return t1, t2, label, p.get("finger_id_1", ""), p.get("finger_id_2", "")


# Sanity check
_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_test.csv", PROJECT_ROOT, IMG_SIZE_TRM, max_pairs=16
)
x1, x2, lbl, _, _ = _ds[0]
print("Sanity check — img shape:", x1.shape, "label:", lbl)

## Verification Metrics

Same helper as in the baseline notebook — computes FAR, FRR, EER by sweeping thresholds.

In [ ]:
# Threshold sweep → FAR, FRR, EER, accuracy
def compute_verification_metrics(
    labels: np.ndarray, scores: np.ndarray, num_thresholds: int = 500
) -> dict:
    thresholds = np.linspace(float(scores.min()), float(scores.max()), num_thresholds)
    genuine  = labels == 1
    impostor = labels == 0
    n_gen = int(genuine.sum())
    n_imp = int(impostor.sum())

    fars, frrs = [], []
    for t in thresholds:
        pred_match = scores >= t
        fars.append(float((pred_match & impostor).sum() / max(n_imp, 1)))
        frrs.append(float((~pred_match & genuine).sum() / max(n_gen, 1)))

    fars = np.array(fars)
    frrs = np.array(frrs)
    eer_idx  = int(np.argmin(np.abs(fars - frrs)))
    eer      = float((fars[eer_idx] + frrs[eer_idx]) / 2)
    eer_thresh = float(thresholds[eer_idx])
    preds_at_eer = (scores >= eer_thresh).astype(int)
    accuracy = float(accuracy_score(labels, preds_at_eer))

    return {
        "eer": eer,
        "eer_threshold": eer_thresh,
        "accuracy_at_eer": accuracy,
        "far_at_eer": float(fars[eer_idx]),
        "frr_at_eer": float(frrs[eer_idx]),
        "thresholds": thresholds,
        "fars": fars,
        "frrs": frrs,
    }


def plot_far_frr(metrics: dict, title: str, save_path: Path | None = None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(metrics["thresholds"], metrics["fars"], label="FAR", color="red")
    ax.plot(metrics["thresholds"], metrics["frrs"], label="FRR", color="blue")
    ax.axvline(
        metrics["eer_threshold"], ls="--", color="gray",
        label=f'EER = {metrics["eer"]:.4f} @ τ={metrics["eer_threshold"]:.3f}',
    )
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Error rate")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
        print("Saved:", save_path.name)
    plt.show()
    return fig


print("Metrics helpers ready")

## Model Architecture — SiamViTRM

**Design (increased capacity):**
- `FingerprintPatchEmbed`: 128×128 → 64 patches of 16×16 → linear project to d=192
- `ViTRMEncoder`: shared 4-layer Transformer block applied recursively:
  - `T_recursion=1` (all cycles run with full gradient — ensures y_init/z_init are trainable)
  - Each cycle: `n_latent_steps=3` z-updates + 1 y-update
  - K=24 latent memory tokens (up from 16)
  - Effective depth: `1 × (3+1) × 4 = 16` effective layers through the same 4-layer block
- `SiamViTRM`: Siamese structure — both images share the same encoder
  - Verification score: cosine similarity between prediction tokens y1, y2
  - Halting head: confidence that current score is correct
- ~1.8M trainable parameters (up from 646K, still ~6× fewer than Siamese ResNet-18)

In [ ]:
# Instantiate SiamViTRM and count parameters
model = SiamViTRM(
    img_size       = 128,   # fingerprint images resized to 128×128
    patch_size     = 16,    # 16×16 patches → 64 tokens per image
    d_model        = 192,   # embedding dimension (was 128 — increased capacity)
    n_heads        = 6,     # attention heads (head_dim = 32)
    n_blocks       = 4,     # layers in the shared Transformer block (was 3)
    K              = 24,    # latent memory tokens (was 16)
    n_latent_steps = 3,     # z-update steps per recursion cycle (ViTRM optimal)
    T_recursion    = 1,     # all-gradient recursion (no no-grad warmup) — ensures y_init/z_init are trained
).to(DEVICE)

total_params = count_parameters(model)
print(f"SiamViTRM total trainable parameters: {total_params:,}")
print(f"  vs Siamese ResNet-18 baseline:       11,242,176")
print(f"  Parameter reduction factor:          {11_242_176 / total_params:.1f}×")
print()

# Architecture breakdown
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>8,} params")

In [ ]:
# Quick forward pass sanity check
model.eval()
with torch.no_grad():
    dummy1 = torch.randn(4, 1, 128, 128).to(DEVICE)
    dummy2 = torch.randn(4, 1, 128, 128).to(DEVICE)
    score, q, y1d, z1d, y2d, z2d, v1, v2 = model(dummy1, dummy2)

print("score shape:", score.shape, "  range:", score.min().item(), "-", score.max().item())
print("q shape:    ", q.shape,     "  range:", q.min().item(), "-", q.max().item())
print("y1_det:     ", y1d.shape)
print("z1_det:     ", z1d.shape)
print("v1:         ", v1.shape)
model.train()

## Training Setup

Aligned with `SiamViTRM_Colab_Final.ipynb` (same loss, schedule, and hyperparameters unless noted):

- **All 17,248 training pairs**; validation uses all 3,696 val pairs; test uses all 3,696 pairs when `TEST_PAIRS_CAP` is `None`.
- **Augmentation (train only):** rotation $\pm 15^\circ$, translation $\pm 8$ px, brightness scale $\pm 25\%$, paired horizontal flip with $p{=}0.5$ (same decision on both images).
- **Optimizer:** AdamW, peak LR `LR`, $\beta{=}(0.9, 0.95)$, weight decay `WEIGHT_DECAY`.
- **Schedule:** 10% linear warmup, then cosine decay to `MIN_LR_FRAC` of peak LR.
- **Training loop:** batch size 32, 20 epochs, grad clip 1.0, EMA decay 0.995; checkpoint by **best validation EER** on the EMA model. Validation loss logged is **BCE on the score only** (same as Colab).
- **Loss:** `BCE(score, label) + 0.5 * CosineEmbeddingLoss(v1, v2)` with margin 0.3.

**Deep supervision:** `N_SUPERVISION` forward+backward passes per batch; detached $(y,z)$ warm-start the next pass. Default `N_SUPERVISION=1`. An optional ablation over `{1, 2}` appears later (`RUN_ABLATION`).

**Local vs Colab:** DataLoaders use `num_workers=0` and `pin_memory` only on CUDA (macOS/CPU-safe). On GPU, set `num_workers=2` if desired to match Colab throughput.

In [ ]:
# Hyperparameters — match SiamViTRM_Colab_Final.ipynb (see Training Setup markdown)
TRAIN_PAIRS_CAP = None    # all 17,248 training pairs
VAL_PAIRS_CAP   = None    # all 3,696 val pairs
TEST_PAIRS_CAP  = None    # all 3,696 test pairs (Colab uses None in eval; cap only for quick tests)
BATCH_SIZE      = 32
EPOCHS          = 20
LR              = 1e-4    # final run + artifacts/trm/trm_results.json
WEIGHT_DECAY    = 0.05    # was missing; required by AdamW (same as Colab)
N_SUPERVISION   = 1
EMA_DECAY       = 0.995
GRAD_CLIP       = 1.0

# ── Loss function (Colab): BCE + 0.5 * CosineEmbeddingLoss ───────────────────
# Encourages the prediction-token embeddings to align for genuine pairs and
# separate for impostor pairs, while still training the verification score.

def compute_step_loss(
    score: torch.Tensor,
    labels: torch.Tensor,
    v1: torch.Tensor,
    v2: torch.Tensor,
) -> torch.Tensor:
    bce = F.binary_cross_entropy(score, labels.float())
    lbl_signed = labels.float() * 2.0 - 1.0  # {0,1} → {-1,+1}
    cel = F.cosine_embedding_loss(v1, v2, lbl_signed, margin=0.3)
    return bce + 0.5 * cel

print("Loss function: BCE(score,label) + 0.5*cosine_embedding(v1,v2)")

# Datasets — augmentation enabled for training only
print("Loading train set...")
train_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_train.csv", PROJECT_ROOT, IMG_SIZE_TRM,
    max_pairs=TRAIN_PAIRS_CAP,
    augment=True,
)
print("Loading val set...")
val_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_val.csv", PROJECT_ROOT, IMG_SIZE_TRM,
    max_pairs=VAL_PAIRS_CAP,
    augment=False,
)

# Local-friendly dataloader settings (safe on macOS / CPU)
PIN_MEMORY = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY
)

# Optimizer — AdamW with β2=0.95 as in TRM paper
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY
)

# Cosine LR schedule with linear warmup + non-zero min LR (same as Colab main loop)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, int(0.10 * total_steps))
MIN_LR_FRAC  = 0.05

def get_lr_scale(step: int) -> float:
    if step < warmup_steps:
        return step / warmup_steps
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return MIN_LR_FRAC + (1.0 - MIN_LR_FRAC) * 0.5 * (1.0 + math.cos(math.pi * p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_scale)

# EMA shadow model for evaluation
ema = EMA(model, decay=EMA_DECAY)

print(f"\nTrain pairs: {len(train_ds)},  Val: {len(val_ds)}")
print(f"Batches/epoch: {len(train_loader)},  Total steps: {total_steps},  Warmup: {warmup_steps}")

## Training Loop

Each training batch:

1. Forward SiamViTRM (possibly multiple times if `N_SUPERVISION > 1`), carrying detached `(y,z)` between supervision steps.
2. **Loss:** `compute_step_loss` = BCE on the match score plus $0.5 \times$ cosine embedding loss on $(v_1, v_2)$.
3. Backward, grad clip, **AdamW** step, **LR scheduler** step, **EMA** update (same order as Colab).

Checkpointing: **best validation EER** on the EMA model (not val loss). After training, the best EMA weights are reloaded before test evaluation.

In [ ]:
# Training loop with deep supervision and EMA (matches Colab logic)
if DEVICE.type == "cuda":
    import os

    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    torch.cuda.empty_cache()

train_losses, val_losses = [], []
global_step = 0

best_val_eer = float("inf")
best_ema_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    epoch_loss = 0.0
    n_batches  = len(train_loader)

    for bi, (img1, img2, labels, _, _) in enumerate(train_loader):
        img1   = img1.to(DEVICE)
        img2   = img2.to(DEVICE)
        labels = labels.to(DEVICE)

        # Deep supervision: initialize states to None (→ learned y_init, z_init)
        y1, z1, y2, z2 = None, None, None, None
        batch_loss = 0.0

        for sup_step in range(N_SUPERVISION):
            score, q, y1, z1, y2, z2, v1, v2 = model(img1, img2, y1, z1, y2, z2)
            step_loss = compute_step_loss(score, labels, v1, v2)
            batch_loss += step_loss.item()
            optimizer.zero_grad()
            step_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            ema.update(model)
            global_step += 1

        epoch_loss += batch_loss / N_SUPERVISION

        if (bi + 1) % 50 == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(
                f"  Epoch {epoch} batch {bi + 1}/{n_batches} "
                f"loss={batch_loss / N_SUPERVISION:.4f} lr={lr_now:.2e}"
            )

    train_losses.append(epoch_loss / n_batches)

    # Validation: BCE loss for reporting + EER for checkpoint selection
    ema.eval_model.eval()
    vloss = 0.0
    val_scores_list: list[np.ndarray] = []
    val_labels_list: list[np.ndarray] = []

    with torch.no_grad():
        for img1, img2, labels, _, _ in val_loader:
            img1   = img1.to(DEVICE)
            img2   = img2.to(DEVICE)
            labels = labels.to(DEVICE)
            score, q, *_ = ema.eval_model(img1, img2)
            vloss += F.binary_cross_entropy(score, labels.float()).item()
            val_scores_list.append(score.detach().cpu().numpy())
            val_labels_list.append(labels.detach().cpu().numpy())

    val_losses.append(vloss / len(val_loader))

    val_scores_arr = np.concatenate(val_scores_list)
    val_labels_arr = np.concatenate(val_labels_list)
    val_eer = compute_verification_metrics(val_labels_arr, val_scores_arr)["eer"]

    if val_eer < best_val_eer:
        best_val_eer = val_eer
        best_ema_state = copy.deepcopy(ema.eval_model.state_dict())
        ckpt_ema_best = TRM_DIR / "siamvitrm_ema_best.pt"
        torch.save(best_ema_state, ckpt_ema_best)
        print(f"  ✓ New best val EER: {best_val_eer * 100:.2f}% — saving EMA checkpoint")

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"train_loss={train_losses[-1]:.4f}  "
        f"val_loss={val_losses[-1]:.4f}  "
        f"val_eer={val_eer * 100:.2f}%  "
        f"({elapsed:.1f}s)"
    )

if best_ema_state is not None:
    ema.eval_model.load_state_dict(best_ema_state)
    print(f"\nRestored best EMA checkpoint (val_eer={best_val_eer * 100:.2f}%)")

# Save model checkpoints
ckpt_main = TRM_DIR / "siamvitrm_main.pt"
ckpt_ema  = TRM_DIR / "siamvitrm_ema.pt"
torch.save(model.state_dict(), ckpt_main)
torch.save(ema.eval_model.state_dict(), ckpt_ema)
print("\nSaved:", ckpt_main.name)
print("Saved:", ckpt_ema.name)
print("Saved:", "siamvitrm_ema_best.pt")

In [ ]:
# Plot training and validation loss curves
fig, ax = plt.subplots(figsize=(8, 4))
epochs_x = range(1, EPOCHS + 1)
ax.plot(epochs_x, train_losses, marker="o", label="Train",     color="steelblue")
ax.plot(epochs_x, val_losses,   marker="s", label="Val (EMA)", color="darkorange")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("SiamViTRM — Training and Validation Loss")
ax.legend()
fig.tight_layout()
save_path = FIG_DIR / "siamvitrm_training_curve.png"
fig.savefig(save_path)
plt.show()
print("Saved:", save_path.name)

## Evaluation on Test Set

Evaluation uses the **EMA model** (more stable than the main model).
Scores are computed on **all test pairs** (matches the Colab notebook).

In [ ]:
# Load test set and run EMA model (matches Colab: evaluate on ALL test pairs)
print("Loading test set...")
test_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_test.csv", PROJECT_ROOT, IMG_SIZE_TRM,
    max_pairs=TEST_PAIRS_CAP,  # None → all pairs
)
PIN_MEMORY = DEVICE.type == "cuda"
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY
)

ema.eval_model.eval()
all_scores: list[np.ndarray] = []
all_labels: list[np.ndarray] = []

with torch.no_grad():
    for img1, img2, labels, _, _ in test_loader:
        img1 = img1.to(DEVICE)
        img2 = img2.to(DEVICE)
        score, _, *_ = ema.eval_model(img1, img2)
        all_scores.append(score.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

trm_scores = np.concatenate(all_scores)
trm_labels = np.concatenate(all_labels)
print(f"Evaluated {len(trm_scores)} test pairs")
print(f"Score range: [{trm_scores.min():.4f}, {trm_scores.max():.4f}]")

In [ ]:
# Compute verification metrics
trm_metrics = compute_verification_metrics(trm_labels, trm_scores)

print("=" * 50)
print("SiamViTRM Test Results")
print("=" * 50)
print(f"  EER:          {trm_metrics['eer']:.4f}  ({trm_metrics['eer'] * 100:.2f}%)")
print(f"  Accuracy:     {trm_metrics['accuracy_at_eer']:.4f}")
print(f"  FAR @ EER:    {trm_metrics['far_at_eer']:.4f}")
print(f"  FRR @ EER:    {trm_metrics['frr_at_eer']:.4f}")
print(f"  Threshold:    {trm_metrics['eer_threshold']:.4f}")
print()

In [ ]:
# Plot FAR vs FRR curve
plot_far_frr(
    trm_metrics,
    "FAR vs FRR — SiamViTRM (FVC2004)",
    FIG_DIR / "far_frr_siamvitrm.png",
)

In [ ]:
# Score distribution: genuine vs impostor
genuine_scores  = trm_scores[trm_labels == 1]
impostor_scores = trm_scores[trm_labels == 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(genuine_scores,  bins=50, alpha=0.7, label="Genuine",  color="steelblue",  density=True)
ax.hist(impostor_scores, bins=50, alpha=0.7, label="Impostor", color="darkorange", density=True)
ax.axvline(
    trm_metrics["eer_threshold"], ls="--", color="gray",
    label=f'EER threshold = {trm_metrics["eer_threshold"]:.3f}',
)
ax.set_xlabel("Verification Score")
ax.set_ylabel("Density")
ax.set_title("SiamViTRM — Score Distribution (Genuine vs Impostor)")
ax.legend()
fig.tight_layout()
save_path = FIG_DIR / "siamvitrm_score_distribution.png"
fig.savefig(save_path)
plt.show()
print("Saved:", save_path.name)

## Error Analysis

In [ ]:
# Identify false accepts and false rejects at EER threshold
threshold  = trm_metrics["eer_threshold"]
predicted  = (trm_scores >= threshold).astype(int)
eval_pairs = test_ds.pairs[: len(trm_labels)]

false_accepts: list[dict] = []
false_rejects: list[dict] = []

for i, p in enumerate(eval_pairs):
    true_lbl = int(p["label"])
    pred_lbl = int(predicted[i])
    if pred_lbl == 1 and true_lbl == 0:
        false_accepts.append(p)
    elif pred_lbl == 0 and true_lbl == 1:
        false_rejects.append(p)

print(f"False accepts (impostors misclassified as genuine): {len(false_accepts)}")
print(f"False rejects (genuines misclassified as impostors): {len(false_rejects)}")
print(f"Total errors: {len(false_accepts) + len(false_rejects)} / {len(trm_labels)}")
print(f"Error rate:   {(len(false_accepts) + len(false_rejects)) / len(trm_labels):.4f}")

In [ ]:
# Cross-partition error breakdown (sensor mismatch analysis)
def get_partition(finger_id: str) -> str:
    """Extract DB partition name from finger_id (e.g. 'DB2_A_040' → 'DB2_A')."""
    parts = finger_id.rsplit("_", 1)
    return parts[0] if len(parts) == 2 else finger_id


cross_partition_fa = sum(
    1 for p in false_accepts
    if get_partition(p.get("finger_id_1", "")) != get_partition(p.get("finger_id_2", ""))
)
cross_partition_fr = sum(
    1 for p in false_rejects
    if get_partition(p.get("finger_id_1", "")) != get_partition(p.get("finger_id_2", ""))
)

print(f"Cross-partition false accepts: {cross_partition_fa} / {len(false_accepts)} "
      f"({100 * cross_partition_fa / max(len(false_accepts), 1):.1f}%)")
print(f"Cross-partition false rejects: {cross_partition_fr} / {len(false_rejects)} "
      f"({100 * cross_partition_fr / max(len(false_rejects), 1):.1f}%)")

## Ablation — `N_supervision` (optional short run)

Quick check whether a second supervised unrolling step per batch changes test EER, using the **same architecture and loss** as the main Colab/local run (`compute_step_loss`, cosine schedule with warmup scaled by `N_supervision` per batch).

For a full **LR × N_supervision** sweep (20 epochs per config), use the **Optional: LR × N_sup grid search** section before **Save Results** (`RUN_GRID_SEARCH = True`).

> **Note:** Trains a fresh model; can be slow on CPU. Skip with `RUN_ABLATION = False`.

In [ ]:
# Ablation: N_supervision = 2
RUN_ABLATION  = False
ABLATION_EPOCHS = 10   # shorter run for ablation

ablation_results: dict[int, dict] = {}

if RUN_ABLATION:
    for n_sup in [2]:
        print(f"\n{'='*50}")
        print(f"Ablation: N_supervision = {n_sup}")
        print(f"{'='*50}")

        torch.manual_seed(SEED)
        abl_model = SiamViTRM(
            img_size=128, patch_size=16, d_model=192, n_heads=6,
            n_blocks=4, K=24, n_latent_steps=3, T_recursion=1,
        ).to(DEVICE)
        abl_ema = EMA(abl_model, decay=EMA_DECAY)
        abl_opt = torch.optim.AdamW(
            abl_model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY
        )
        abl_total_steps = len(train_loader) * ABLATION_EPOCHS * n_sup
        abl_warmup = max(1, int(0.10 * abl_total_steps))

        def _abl_lr_fn(step: int) -> float:
            if step < abl_warmup:
                return step / abl_warmup
            p = (step - abl_warmup) / max(1, abl_total_steps - abl_warmup)
            return MIN_LR_FRAC + (1.0 - MIN_LR_FRAC) * 0.5 * (1.0 + math.cos(math.pi * p))

        abl_sched = torch.optim.lr_scheduler.LambdaLR(abl_opt, lr_lambda=_abl_lr_fn)

        for epoch in range(1, ABLATION_EPOCHS + 1):
            t0 = time.time()
            abl_model.train()
            epoch_loss = 0.0
            n_b = len(train_loader)

            for img1, img2, labels, _, _ in train_loader:
                img1   = img1.to(DEVICE)
                img2   = img2.to(DEVICE)
                labels = labels.to(DEVICE)

                y1, z1, y2, z2 = None, None, None, None
                batch_loss = 0.0

                for sup_step in range(n_sup):
                    sc, q, y1, z1, y2, z2, v1, v2 = abl_model(img1, img2, y1, z1, y2, z2)
                    step_loss = compute_step_loss(sc, labels, v1, v2)
                    batch_loss += step_loss.item()
                    abl_opt.zero_grad()
                    step_loss.backward()
                    torch.nn.utils.clip_grad_norm_(abl_model.parameters(), GRAD_CLIP)
                    abl_opt.step()
                    abl_sched.step()
                    abl_ema.update(abl_model)

                epoch_loss += batch_loss / n_sup

            print(f"  Epoch {epoch}/{ABLATION_EPOCHS} loss={epoch_loss / n_b:.4f} ({time.time()-t0:.1f}s)")

        # Evaluate ablation model
        abl_ema.eval_model.eval()
        abl_scores_list: list[np.ndarray] = []
        with torch.no_grad():
            for img1, img2, labels, _, _ in test_loader:
                img1 = img1.to(DEVICE)
                img2 = img2.to(DEVICE)
                sc, _, *_ = abl_ema.eval_model(img1, img2)
                abl_scores_list.append(sc.cpu().numpy())

        abl_scores  = np.concatenate(abl_scores_list)
        abl_metrics = compute_verification_metrics(trm_labels, abl_scores)
        ablation_results[n_sup] = {"eer": abl_metrics["eer"], "accuracy": abl_metrics["accuracy_at_eer"]}
        print(f"  N_sup={n_sup} → EER={abl_metrics['eer']:.4f}")
else:
    print("Ablation skipped (RUN_ABLATION=False).")

## Comparison with Baselines

In [ ]:
# Load baseline results
with open(ARTIFACT_DIR / "baseline_results.json") as f:
    baseline = json.load(f)

ncc_eer  = baseline["ncc_baseline"]["eer"]
ncc_acc  = baseline["ncc_baseline"]["accuracy_at_eer"]
rn18_eer = baseline["siamese_resnet18_baseline"]["eer"]
rn18_acc = baseline["siamese_resnet18_baseline"]["accuracy_at_eer"]
rn18_par = baseline["siamese_resnet18_baseline"]["model_params"]

trm_eer  = trm_metrics["eer"]
trm_acc  = trm_metrics["accuracy_at_eer"]
trm_par  = count_parameters(model)

print(f"Comparison Table — FVC2004 Test Set ({len(trm_scores):,} pairs)")
print("=" * 70)
print(f"{'Model':<28} {'Params':>10} {'EER':>8} {'Acc@EER':>10} {'vs ResNet-18':>14}")
print("-" * 70)
print(f"{'NCC (no learning)':<28} {'0':>10} {ncc_eer:>8.4f} {ncc_acc:>10.4f} {'—':>14}")
print(f"{'Siamese ResNet-18':<28} {rn18_par:>10,} {rn18_eer:>8.4f} {rn18_acc:>10.4f} {'baseline':>14}")
print(f"{'SiamViTRM (ours)':<28} {trm_par:>10,} {trm_eer:>8.4f} {trm_acc:>10.4f} "
      f"{rn18_par / trm_par:>11.1f}× fewer")
print("=" * 70)

if ablation_results:
    print("\nAblation — N_supervision")
    print(f"  N_sup=1 (primary):  EER={trm_eer:.4f}")
    for n_sup, res in sorted(ablation_results.items()):
        print(f"  N_sup={n_sup}:            EER={res['eer']:.4f}")

In [ ]:
# Overlay all FAR/FRR curves on one plot for easy comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
titles  = ["NCC (no learning)", "Siamese ResNet-18", "SiamViTRM (ours)"]
eers    = [ncc_eer, rn18_eer, trm_eer]
colors  = ["gray", "steelblue", "darkorange"]

# Load NCC and ResNet metrics from saved results for FAR/FRR arrays
# (those arrays aren't in the JSON — re-derive from score distributions if needed;
#  here we just show the TRM curve with overlay lines for the other EER values)
for ax, title, eer, color in zip(axes, titles, eers, colors):
    if title == "SiamViTRM (ours)":
        ax.plot(trm_metrics["thresholds"], trm_metrics["fars"], label="FAR", color="red")
        ax.plot(trm_metrics["thresholds"], trm_metrics["frrs"], label="FRR", color="blue")
        ax.axvline(trm_metrics["eer_threshold"], ls="--", color="gray", label=f"EER={eer:.3f}")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, f"EER = {eer:.4f}", ha="center", va="center",
                transform=ax.transAxes, fontsize=14, color=color)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Threshold")

axes[0].set_ylabel("Error rate")
fig.suptitle("Fingerprint Verification — Model Comparison (FVC2004)", fontsize=12)
fig.tight_layout()
save_path = FIG_DIR / "comparison_far_frr.png"
fig.savefig(save_path)
plt.show()
print("Saved:", save_path.name)

## Optional: LR $\times$ $N_\mathrm{sup}$ grid search (Colab parity)

Same sweep as `SiamViTRM_Colab_Final.ipynb`: peak LR $\in\{10^{-4},\,3{\times}10^{-4},\,6{\times}10^{-4}\}$ and $N_\mathrm{sup}\in\{1,2\}$ (20 epochs each, fresh model per config). Scheduler total steps scale with $N_\mathrm{sup}$ per batch (as in the Colab grid cell). **Run this only after** `train_loader`, `val_loader`, and `test_loader` exist (i.e., after training + test `DataLoader` cells). Set `RUN_GRID_SEARCH = True` to execute (very long on CPU).

In [ ]:
# Optional grid search — set RUN_GRID_SEARCH = True to run (GPU strongly recommended)
RUN_GRID_SEARCH = False
GS_LR_VALUES = [1e-4, 3e-4, 6e-4]
GS_NSUP_VALUES = [1, 2]
GS_EPOCHS = 20
BASELINE_VAL_EER = 17.61  # % — historical 3e-4, N_sup=1 val anchor (Colab); update if you re-baseline
BASELINE_TEST_EER = 15.88  # % — historical test anchor before final 1e-4 run

if not RUN_GRID_SEARCH:
    print("Grid search skipped (RUN_GRID_SEARCH = False). Set True to match Colab sweep.")
else:
    assert "test_loader" in globals(), "Run test evaluation cell first so test_loader exists."
    grid_results = []
    for lr_val in GS_LR_VALUES:
        for nsup_val in GS_NSUP_VALUES:
            print(f"\n{'='*62}\n  Config: lr={lr_val:.0e}  N_SUPERVISION={nsup_val}\n{'='*62}")
            torch.manual_seed(SEED)
            gs_model = SiamViTRM(
                img_size=128, patch_size=16, d_model=192, n_heads=6,
                n_blocks=4, K=24, n_latent_steps=3, T_recursion=1,
            ).to(DEVICE)
            gs_ema = EMA(gs_model, decay=EMA_DECAY)
            gs_optimizer = torch.optim.AdamW(
                gs_model.parameters(), lr=lr_val, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY
            )
            gs_total_steps = len(train_loader) * GS_EPOCHS * nsup_val
            gs_warmup_steps = max(1, int(0.10 * gs_total_steps))

            def _make_lr_fn(total, warmup):
                def fn(step):
                    if step < warmup:
                        return step / warmup
                    p = (step - warmup) / max(1, total - warmup)
                    return MIN_LR_FRAC + (1.0 - MIN_LR_FRAC) * 0.5 * (1.0 + math.cos(math.pi * p))
                return fn

            gs_scheduler = torch.optim.lr_scheduler.LambdaLR(
                gs_optimizer, lr_lambda=_make_lr_fn(gs_total_steps, gs_warmup_steps)
            )
            best_gs_val_eer = float("inf")
            best_gs_ema_state = None
            for epoch in range(1, GS_EPOCHS + 1):
                t0 = time.time()
                gs_model.train()
                epoch_loss = 0.0
                for img1, img2, labels, _, _ in train_loader:
                    img1 = img1.to(DEVICE)
                    img2 = img2.to(DEVICE)
                    labels = labels.to(DEVICE)
                    y1, z1, y2, z2 = None, None, None, None
                    batch_loss = 0.0
                    for _ in range(nsup_val):
                        score, q, y1, z1, y2, z2, v1, v2 = gs_model(img1, img2, y1, z1, y2, z2)
                        step_loss = compute_step_loss(score, labels, v1, v2)
                        batch_loss += step_loss.item()
                        gs_optimizer.zero_grad()
                        step_loss.backward()
                        torch.nn.utils.clip_grad_norm_(gs_model.parameters(), GRAD_CLIP)
                        gs_optimizer.step()
                        gs_scheduler.step()
                        gs_ema.update(gs_model)
                    epoch_loss += batch_loss / nsup_val
                gs_ema.eval_model.eval()
                vs_list, vl_list = [], []
                with torch.no_grad():
                    for img1, img2, labels, _, _ in val_loader:
                        img1 = img1.to(DEVICE)
                        img2 = img2.to(DEVICE)
                        labels = labels.to(DEVICE)
                        score, q, *_ = gs_ema.eval_model(img1, img2)
                        vs_list.append(score.cpu().numpy())
                        vl_list.append(labels.cpu().numpy())
                val_eer = compute_verification_metrics(
                    np.concatenate(vl_list), np.concatenate(vs_list)
                )["eer"]
                if val_eer < best_gs_val_eer:
                    best_gs_val_eer = val_eer
                    best_gs_ema_state = copy.deepcopy(gs_ema.eval_model.state_dict())
                    tag = " <- best"
                else:
                    tag = ""
                print(
                    f"  Epoch {epoch:2d}/{GS_EPOCHS}  train_loss={epoch_loss / len(train_loader):.4f}  "
                    f"val_eer={val_eer * 100:.2f}%{tag}  ({time.time() - t0:.1f}s)"
                )
            print("\n  Evaluating best checkpoint on test set...")
            gs_ema.eval_model.load_state_dict(best_gs_ema_state)
            gs_ema.eval_model.eval()
            ts_list, tl_list = [], []
            with torch.no_grad():
                for img1, img2, labels, _, _ in test_loader:
                    img1 = img1.to(DEVICE)
                    img2 = img2.to(DEVICE)
                    score, q, *_ = gs_ema.eval_model(img1, img2)
                    ts_list.append(score.cpu().numpy())
                    tl_list.append(labels.cpu().numpy())
            test_eer = compute_verification_metrics(
                np.concatenate(tl_list), np.concatenate(ts_list)
            )["eer"]
            print(f"  Test EER: {test_eer * 100:.2f}%")
            grid_results.append({
                "lr": lr_val,
                "N_SUPERVISION": nsup_val,
                "best_val_eer_pct": round(best_gs_val_eer * 100, 3),
                "test_eer_pct": round(test_eer * 100, 3),
            })
            del gs_model, gs_ema, gs_optimizer, gs_scheduler
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    W = 80
    print("\n" + "=" * W)
    print("  GRID SEARCH RESULTS (sorted by best val EER)")
    print(f"  Reference: lr=3e-4, N_SUP=1  val~{BASELINE_VAL_EER:.2f}%  test~{BASELINE_TEST_EER:.2f}%")
    print("=" * W)
    print(f"  {'LR':>8}  {'N_SUP':>5}  {'Best Val EER':>14}  {'Test EER':>10}  {'Δ val vs ref':>14}")
    print("  " + "-" * (W - 2))
    for r in sorted(grid_results, key=lambda x: x["best_val_eer_pct"]):
        dv = r["best_val_eer_pct"] - BASELINE_VAL_EER
        print(
            f"  {r['lr']:>8.0e}  {r['N_SUPERVISION']:>5}  "
            f"{r['best_val_eer_pct']:>12.2f}%  {r['test_eer_pct']:>8.2f}%  {dv:>+13.2f}%"
        )
    print("=" * W)


## Save Results

In [ ]:
# Save TRM results to artifacts/trm/trm_results.json
trm_results = {
    "model": "SiamViTRM",
    "architecture": {
        "img_size": 128,
        "patch_size": 16,
        "n_patches": 64,
        "d_model": 192,
        "n_heads": 6,
        "n_blocks": 4,
        "K": 24,
        "n_latent_steps": 3,
        "T_recursion": 1,
        "effective_depth": 1 * (3 + 1) * 4,  # T_recursion * (n_latent_steps+1) * n_blocks
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "N_supervision": N_SUPERVISION,
        "EMA_decay": EMA_DECAY,
        "train_pairs": len(train_ds),
        "loss_function": "BCE",
        "pretrained": False,
    },
    "evaluation": {
        "test_pairs": int(len(trm_scores)),
        "eer": float(trm_metrics["eer"]),
        "eer_threshold": float(trm_metrics["eer_threshold"]),
        "accuracy_at_eer": float(trm_metrics["accuracy_at_eer"]),
        "far_at_eer": float(trm_metrics["far_at_eer"]),
        "frr_at_eer": float(trm_metrics["frr_at_eer"]),
        "model_params": int(count_parameters(model)),
        "false_accepts": len(false_accepts),
        "false_rejects": len(false_rejects),
    },
    "baselines": {
        "ncc": {"eer": float(ncc_eer), "params": 0},
        "siamese_resnet18": {"eer": float(rn18_eer), "params": int(rn18_par)},
    },
    "ablation_n_supervision": {
        str(k): v for k, v in ablation_results.items()
    } if ablation_results else {},
}

out_path = TRM_DIR / "trm_results.json"
with open(out_path, "w") as f:
    json.dump(trm_results, f, indent=2)

print(f"Results saved to: {out_path}")
print()
print(json.dumps(trm_results, indent=2))